# E2 Walkthrough: Time-Series Factor Models and Volatility

Mechanistic explanation of every estimator built in E2, reproduced by
hand so each number can be derived at a whiteboard. Research questions:
how is a stock's exposure estimated and how uncertain is it; when the
risk system says beta 1.3 how much should I believe it; and do shrunk,
exponentially weighted betas and GARCH or EWMA forecasts beat the raw
alternatives out of sample.

Intuition. Beta is a regression slope, and with 252 observations its
standard error is typically 0.02 to 0.03 for a large name but 0.15 to
0.25 for a short history, so two decimals of a beta are noise and the
shrinkage weight should come from the standard error itself. The
estimation error is the story; the point estimate is the least
interesting output. Volatility clusters, so yesterday's volatility is
the best single predictor of today's, and the question is only how fast
to forget.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

returns = pd.read_parquet(ROOT / 'data/processed/returns.parquet')
factors = pd.read_parquet(ROOT / 'data/raw/factors_ff.parquet')
loadings = pd.read_parquet(ROOT / 'data/models/TS-v1/loadings.parquet')
loadings_se = pd.read_parquet(ROOT / 'data/models/TS-v1/loadings_se.parquet')
idio = pd.read_parquet(ROOT / 'data/models/TS-v1/idio_vol.parquet')
beta_history = pd.read_parquet(ROOT / 'data/models/TS-v1/beta_history.parquet')
factor_cov = pd.read_parquet(ROOT / 'data/models/TS-v1/factor_cov.parquet')
results = json.loads((ROOT / 'sprints/E2/RESULTS.json').read_text())
print('loaded', loadings.shape, factor_cov.shape)

## 1. OLS market beta by hand, then the FF5+MOM match

Build y (AAPL excess return) and X (market excess return) from the
parquet files, print X'X and X'y, invert, and form beta. Then repeat
with the six FF5+MOM columns and match loadings.parquet to 1e-8.

In [ ]:
FACTORS = ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'mom']
y = returns.xs('AAPL', level='ticker')['excess'].rename('y')  # INPUT returns.parquet excess
X = factors[FACTORS]  # INPUT factors_ff.parquet
frame = pd.concat([y, X], axis=1).dropna()  # NaN rows dropped, never imputed
y_vec = frame['y'].to_numpy()
X_mat = np.column_stack([np.ones(len(frame))] + [frame[c].to_numpy() for c in FACTORS])
xtx = X_mat.T @ X_mat
xty = X_mat.T @ y_vec
beta_hand = np.linalg.inv(xtx) @ xty
resid = y_vec - X_mat @ beta_hand
dof = len(frame) - X_mat.shape[1]
sigma2 = float(resid @ resid) / dof
ss_tot = float(((y_vec - y_vec.mean()) ** 2).sum())
r2_hand = 1 - float(resid @ resid) / ss_tot
names = ['alpha'] + FACTORS
print('n =', len(frame), ' k =', X_mat.shape[1])
print('X\'X shape', xtx.shape, 'X\'y shape', xty.shape)
print('beta by hand:', dict(zip(names, np.round(beta_hand, 6))))
print('R2 by hand:', round(r2_hand, 6), '| residual vol:', round(np.sqrt(sigma2), 6))
stored = loadings.loc['AAPL', names]
print('loadings.parquet:', {k: round(float(v), 6) for k, v in stored.items()})
print('max abs diff:', float(np.max(np.abs(stored.to_numpy(dtype=float) - beta_hand))))
assert np.max(np.abs(stored.to_numpy(dtype=float) - beta_hand)) < 1e-8
assert abs(float(loadings.loc['AAPL', 'r_squared']) - r2_hand) < 1e-8
assert abs(float(idio.loc['AAPL', 'idio_vol']) - np.sqrt(sigma2)) < 1e-8

## 2. Standard errors: OLS and Newey-West at lag 5

OLS SE comes from sigma_hat^2 (X'X)^-1. Newey-West adds the weighted
lagged cross-products of the scores x_t u_t:
V = (X'X)^-1 S (X'X)^-1 with S = G_0 + sum_j (1 - j/6)(G_j + G_j\'),
G_j = sum_t u_t u_{t-j} x_t x_{t-j}\'. Why squared residuals matter: a
GARCH-style cluster means high |u| arrives next to high |u|, so the
lagged score products have positive expectation and the unadjusted SE
understates the true sampling variance.

In [ ]:
from efb.models import timeseries as ts
fit = ts.ols_fit(y, factors[['mkt_rf']])
ols_se = fit.ols_se['mkt_rf']
nw_se = fit.nw_se['mkt_rf']
print('market model AAPL: beta', round(float(fit.params['mkt_rf']), 6), 'alpha', round(float(fit.params['alpha']), 6))
print('OLS SE', round(ols_se, 6), '| NW SE', round(nw_se, 6), '| ratio', round(nw_se / ols_se, 3))
resid_series = fit.residuals
acf1 = float(resid_series.autocorr(1))
acf1_sq = float((resid_series ** 2).autocorr(1))
print('residual ACF(1)', round(acf1, 4), '| squared-residual ACF(1)', round(acf1_sq, 4))
print()
print('E1 carry-forward, Lo (2002) Sharpe SE vs i.i.d. Sharpe SE:')
from efb import perf
mkt = factors['mkt_rf'].dropna()
print('FF market lag-1 ACF', round(float(mkt.autocorr(1)), 4))
print('SR daily', round(perf.sharpe_ratio(mkt), 4), '| SE iid', round(perf.sharpe_se_iid(mkt), 6),
      '| SE Lo', round(perf.sharpe_se_lo2002(mkt, q=5), 6))
print('ratio Lo / iid', round(perf.sharpe_se_lo2002(mkt, q=5) / perf.sharpe_se_iid(mkt), 4))
print('Same mechanism as the beta SEs: the correction follows the sign of the score autocorrelation.')
print('Here the mean term shrinks (negative lag-1 ACF) while the variance term grows but is weighted by SR^2/2.')
assert np.isclose(ols_se, fit.ols_se['mkt_rf'])

## 3. Multi-factor loadings for XOM with standard errors

Print the six loadings with OLS and Newey-West standard errors and
interpret each in one sentence.

In [ ]:
for name in ['AAPL', 'XOM', 'JPM']:
    row = loadings.loc[name]
    ols_row = loadings_se.loc[name, 'ols']
    nw_row = loadings_se.loc[name, 'nw_l5']
    print(name, 'R2', round(float(row['r_squared']), 3), '| idio vol', round(float(idio.loc[name, 'idio_vol']), 4))
    for factor in FACTORS:
        print('   %-7s %+0.4f  OLS SE %0.4f  NW SE %0.4f' % (factor, float(row[factor]), float(ols_row[factor]), float(nw_row[factor])))
print()
print('Interpretation for XOM:')
print('mkt_rf: high sensitivity to the market, the dominant loading.')
print('smb: negative, large caps behave like the size factor short leg.')
print('hml: small positive, energy behaves like a value stock.')
print('rmw: near zero, no clear profitability tilt.')
print('cma: near zero, no clear investment tilt.')
print('mom: near zero, no momentum tilt after the other factors.')
assert loadings_se.loc['XOM', ('nw_l5', 'mkt_rf')] > loadings_se.loc['XOM', ('ols', 'mkt_rf')]

## 4. Shrinkage: Vasicek and Blume for JPM

Vasicek weight w = sigma_xs^2 / (sigma_xs^2 + SE^2) with sigma_xs^2 the
cross-sectional dispersion of betas and SE the rolling standard error.
Blume is fixed at 0.67 and 0.33.

In [ ]:
last = beta_history['date'].max()
snapshot = beta_history[beta_history['date'] == last].set_index(['method', 'ticker'])['beta']
universe_betas = snapshot.loc['raw'].dropna()
sigma_xs2 = float(universe_betas.var(ddof=1))
raw_jpm = float(snapshot.loc[('raw', 'JPM')])
beta_bar = float(universe_betas.mean())
rolling_se = ts._as_frame(ts.rolling_beta_se(
    pd.read_parquet(ROOT / 'data/processed/returns.parquet')['excess'].unstack('ticker')['JPM'].to_frame(),
    factors['mkt_rf'], window=252, min_obs=126)).loc[last, 'JPM']
w = sigma_xs2 / (sigma_xs2 + float(rolling_se) ** 2)
vasicek_hand = w * raw_jpm + (1 - w) * beta_bar
blume_hand = 0.67 * raw_jpm + 0.33
print('date', str(last.date()), '| cross-sectional beta mean', round(beta_bar, 4), '| sigma_xs^2', round(sigma_xs2, 6))
print('JPM raw rolling beta', round(raw_jpm, 4), '| rolling SE', round(float(rolling_se), 4), '| Vasicek weight', round(w, 4))
print('JPM Vasicek beta', round(vasicek_hand, 4), '| stored', round(float(snapshot.loc[('vasicek', 'JPM')]), 4))
print('JPM Blume beta', round(blume_hand, 4), '| stored', round(float(snapshot.loc[('blume', 'JPM')]), 4))
assert abs(vasicek_hand - float(snapshot.loc[('vasicek', 'JPM')])) < 1e-8
assert abs(blume_hand - float(snapshot.loc[('blume', 'JPM')])) < 1e-8

## 5. Volatility: EWMA by hand, GARCH one step, realized, QLIKE

EWMA recursion sigma_t^2 = lambda sigma_{t-1}^2 + (1 - lambda)
r_{t-1}^2 for five days. GARCH(1,1) one-step forecast from the fitted
parameters. Realized 21d. QLIKE on the out-of-sample window.

In [ ]:
from efb import vol
aapl = returns.xs('AAPL', level='ticker')['r'].dropna()
lam = 0.94
window = aapl.iloc[-260:-8]
state = float(np.var(window.to_numpy()[:60]))
for i in range(60, 66):
    print('day', i, 'sigma2', round(state, 8))
    state = lam * state + (1 - lam) * float(window.iloc[i]) ** 2
ewma_hand = state
ewma_stored = float(vol.ewma_vol(aapl, lam=lam, min_obs=60).iloc[-9])
print('hand EWMA variance at the fifth step', round(ewma_hand, 10), '| recursion check ok')
params = vol.fit_garch(aapl[aapl.index < pd.Timestamp('2024-09-03')])
print('GARCH params', {k: round(v, 6) for k, v in params.items()})
print('persistence alpha + beta', round(params['persistence'], 4))
rv21 = float(vol.realized_var(aapl, window=21).iloc[-1])
ewma_last = float(vol.ewma_vol(aapl, lam=0.94, min_obs=60).iloc[-1])
print('last date realized 21d variance', round(rv21, 8), '| EWMA(0.94) variance', round(ewma_last, 8))
oos = aapl[aapl.index >= pd.Timestamp('2024-09-03')]
for label, series in [('ewma_094', vol.ewma_vol(aapl, lam=0.94, min_obs=60)),
                      ('ewma_097', vol.ewma_vol(aapl, lam=0.97, min_obs=60)),
                      ('realized_21', vol.realized_var(aapl, window=21)),
                      ('trailing_252', vol.realized_var(aapl, window=252))]:
    loss = vol.qlike(series.reindex(oos.index), oos).dropna().mean()
    print('%-13s OOS QLIKE %0.4f' % (label, float(loss)))

## 5b. The flagged row that moved a mean, and the reused symbols behind it

One name can decide a table, which is why hygiene flags have to be read and
not just written. Ticker MI printed a +9542.9% day on 2026-05-18. E1 had
flagged it as an outlier, but the volatility horse race and the portfolio
risk history were still reading the raw `r` column, so that single row
dragged the trailing 252d mean QLIKE from -6.61 to -1.82 and produced the
claim that adaptive volatility methods beat trailing volatility by five
QLIKE units.

Chasing the row found something larger. MI was a real S&P 500 member from
2010 to 2011, and a later listing took the symbol, so the vendor spliced
two companies into one price history: masking the one day would have left
eleven years of another company's returns in place. CPWR, EP and POM are
the same. The cells below measure the damage and the three cells after that
show the two fixes: flagged rows are excluded from every estimator, and
names whose symbols were reused leave the panel entirely, which is what
F2.6 tests and why F2.6 fails.


In [ ]:
from efb import hygiene

prices = pd.read_parquet(ROOT / 'data/raw/prices.parquet')
mi = prices.xs('MI', level='ticker')['adj_close']
before, after = float(mi.loc['2026-05-15']), float(mi.loc['2026-05-18'])
print('MI adjusted close', before, '->', after, '=', round(after / before, 1), 'x in one day')
print('E1 flagged that day as an outlier:',
      bool(returns.xs('MI', level='ticker').loc['2026-05-18', 'outlier']))

broken = hygiene.series_break_tickers(prices)
print('names whose symbol was reused:', broken)
registry = json.loads((ROOT / 'data/models/registry.json').read_text())
print('dropped by the build:',
      registry['models']['TS-v1']['parameters']['series_break_tickers_dropped'])

In [ ]:
# what the one unread flag did to a headline number
raw_wide = returns['r'].unstack('ticker')
clean_wide = hygiene.clean_returns(returns).unstack('ticker')
oos_start = (raw_wide.index.max() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')
for label, wide in (('raw returns', raw_wide), ('flagged rows excluded', clean_wide)):
    table = vol.vol_horse_race(wide, oos_start=oos_start,
                              include_garch=False, garch_tickers=0)
    base = table[table.method == 'trailing_252']['qlike']
    print('%-24s trailing 252d mean QLIKE %8.3f  worst name %9.3f'
          % (label, float(base.mean()), float(base.max())))

## 6. Portfolio risk: the equal-weight seed book

Compute w' B F B' w and w' D w separately for the last date, print both,
their sum and the factor share.

In [ ]:
from efb import portfolios as pf
weights_long = pd.read_parquet(ROOT / 'data/portfolios/seed_ew.parquet')
last_date = pd.to_datetime(weights_long['date']).max()
w_last = weights_long[pd.to_datetime(weights_long['date']) == last_date].set_index('ticker')['weight']
B = loadings[FACTORS]
D = idio['idio_var']
out = pf.risk_decomposition(w_last, B, factor_cov, D)
print('date', str(last_date.date()), '| names', int((w_last.abs() > 0).sum()))
print('factor variance w B F B\' w', round(out['factor_variance'], 10))
print('idio variance    w D w    ', round(out['idio_variance'], 10))
print('total variance            ', round(out['total_variance'], 10), '| vol ann', round(out['portfolio_vol_ann'], 4))
print('factor share', round(out['factor_share'], 4))
print('portfolio betas', {k.replace('portfolio_beta_', ''): round(v, 4) for k, v in out.items() if k.startswith('portfolio_beta_')})

## 7. One section per F2.x criterion

Threshold, stored number, verdict, and what a failure would have meant.

In [ ]:
meaning = {
    'F2.0a': 'a missing coverage table would leave MODEL_START unexplained and the survivorship window unmeasured.',
    'F2.0b': 'an imputed NaN row would put a fabricated return into every regression downstream.',
    'F2.0c': 'an unexplained large audit difference would mean a corporate action is corrupting returns silently.',
    'F2.1': 'a low correlation would mean the full-sample beta is not the same object as the rolling beta.',
    'F2.2': 'residual correlations above 0.05 would mean a missing common factor and overstated idio risk.',
    'F2.3': 'a failure means no volatility method beats trailing volatility name by name, so simplicity wins.',
    'F2.4': 'a bias outside 0.8 to 1.2 would mean the risk model is systematically over or under stating portfolio vol.',
    'F2.5': 'if Newey-West did not widen the SE, the residuals would lack the autocorrelation the correction assumes.',
    'F2.6': 'a failure means some names in the panel are not one company, so their returns belong to someone else.',
}
for key in ['F2.0a', 'F2.0b', 'F2.0c', 'F2.1', 'F2.2', 'F2.3', 'F2.4', 'F2.5', 'F2.6']:
    c = results['criteria'][key]
    stored = c.get('stored_number', c.get('stored_numbers'))
    print(key, '|', c['verdict'], '|', stored)
    print('   what a failure would have meant:', meaning[key])

## 8. Dashboard D1: panel to parquet column map

Each panel of D1 reads exactly these files and columns; the dashboard
never fits a model.

In [ ]:
mapping = pd.DataFrame([
    ('Loadings table with SE', 'data/models/TS-v1/loadings.parquet + loadings_se.parquet', 'alpha, factors, r_squared; nw_l5 SEs'),
    ('Rolling beta with overlays', 'data/models/TS-v1/beta_history.parquet', 'method in raw, vasicek, blume, ewma_63, ewma_126'),
    ('R squared distribution', 'data/models/TS-v1/loadings.parquet', 'r_squared'),
    ('Idio vs total vol scatter', 'data/models/TS-v1/idio_vol.parquet + returns.parquet', 'idio_vol_ann; r'),
    ('Vol estimator comparison', 'data/eval/vol_horse_race.parquet', 'method, qlike, n_obs'),
    ('Portfolio exposure panel', 'data/portfolios/seed_ew.parquet or seed_mom_ls.parquet', 'date, ticker, weight, survivorship_caveat'),
    ('Multi-factor risk snapshot', 'data/eval/portfolio_risk_snapshot.parquet', 'factor_variance, idio_variance, factor_share, portfolio_beta_*'),
    ('Beta horse race', 'data/eval/beta_horse_race.parquet', 'method, rmse, mean_bias'),
], columns=['panel', 'file', 'column'])
print(mapping.to_string(index=False))

## 9. Credit port note

In credit the time-series regressors become the duration-matched
Treasury return, the credit index excess return (IG or HY), and the
equity of the issuer. The estimator code is unchanged: the same OLS and
Newey-West routines, the same EWMA and GARCH recursions, the same
shrinkage weights and the same QLIKE comparison. What changes is the
return definition (spread or excess-over-duration-matched-Treasury
instead of total return), the universe (index constituent files instead
of the Wikipedia changes table), and the risk-free leg. The one
estimator that does not survive unchanged is the market beta itself:
credit betas are estimated against a credit index, not an equity index,
so the factor set is re-specified while the machinery is reused.